<a href="https://colab.research.google.com/github/Abeldb11/Ahadu/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# importing important libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets


In [ ]:
import kagglehub

# Downloading the image datasets
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Path to dataset files: /kaggle/input/intel-image-classification


In [ ]:
from torch.utils.data import Subset
import os
from torch.utils.data import random_split

transform = transforms.Compose([transforms.Resize((150,150)), transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

dataset = datasets.ImageFolder(root=os.path.join(path, "seg_train", "seg_train"), transform=transform)
#print(len(dataset))
train_dir = os.path.join(path, "seg_train", "seg_train")
val_dir   = os.path.join(path, "seg_test",  "seg_test")

n = int(0.8 * len(dataset))
indices = torch.randperm(len(dataset)).tolist()

# now the index themselves are random

# train_set = Subset(dataset, indices[:n])
# test_set = Subset(dataset, indices[n:])

#cleanest way of classification

#train_set, val_set, test_set = random_split(dataset, [0.7,0.2,0.1]) # this line of code can do what randperm plus Subset do together
train_dataset = datasets.ImageFolder(train_dir, transform= transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform= transform)

train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)

print(len(train_dataset))
# print(len(test_dataset))
print(len(val_dataset))


14034
3000


In [ ]:
from torch.utils.data import Subset
from torch.utils.data import random_split
import os
transform = transforms.Compose([transforms.Resize((150,150)), transforms.ToTensor()])
num_classes = len(dataset.classes) # Get the number of classes
#print(len(dataset))


# the model's code without the data, this is the CNN



class convs(nn.Module):
  def __init__(self, num_classes):
    super(convs, self).__init__()
    self.conv1 = nn.Conv2d(3, 16, kernel_size = 3, padding = 1)
    self.conv2 = nn.Conv2d(16, 32, kernel_size = 3, padding = 1)
    self.conv3 = nn.Conv2d(32, 64, kernel_size = 3, padding = 1)
    self.conv4 = nn.Conv2d(64, 128, kernel_size = 3, padding = 1)
    self.conv5 = nn.Conv2d(128, 256, kernel_size = 3, padding = 1)


    self.pool = nn.MaxPool2d(2, 2)
    self.drop2d = nn.Dropout2d(0.1)
    self.dropout = nn.Dropout(0.25)
    # Add a fully connected layer for classification
    # Calculated output size after convolutions and pooling: 64 channels * 17 * 17 pixels
    self.fc = nn.Linear(256 * 4 * 4, 512)
    self.fc2 = nn.Linear(512, num_classes)

    self.bn1 = nn.BatchNorm2d(16)
    self.bn2 = nn.BatchNorm2d(32)
    self.bn3 = nn.BatchNorm2d(64)
    self.bn4 = nn.BatchNorm2d(128)
    self.bn5 = nn.BatchNorm2d(256)

  def forward(self, x):
    x = self.drop2d((self.pool(F.relu(self.bn1(self.conv1(x))))))
    x = self.drop2d((self.pool(F.relu(self.bn2(self.conv2(x))))))
    x = self.drop2d((self.pool(F.relu(self.bn3(self.conv3(x))))))
    x = self.drop2d((self.pool(F.relu(self.bn4(self.conv4(x))))))
    x = self.drop2d((self.pool(F.relu(self.bn5(self.conv5(x))))))



    # Flatten the tensor for the fully connected layer
    x = x.view(-1, 256 * 4 * 4) # -1 infers the batch size
    x = self.dropout(F.relu(self.fc(x)))# Pass through the fully connected layer
    x = self.fc2(x)
    return x

model = convs(num_classes) # Pass num_classes to the model
print("Num classes:", num_classes)

from collections import Counter
labels = [dataset.targets[i] for i in range(len(dataset))]
print("Label distribution:", Counter(labels))
# now we have defined the layers, applied non-linearity and


Num classes: 6
Label distribution: Counter({3: 2512, 2: 2404, 5: 2382, 4: 2274, 1: 2271, 0: 2191})


In [ ]:
if torch.cuda.is_available():
        device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

convs(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2d): Dropout2d(p=0.1, inplace=False)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc): Linear(in_features=4096, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=6, bias=True)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn4): BatchNorm2d(128, eps=1e-05, momentu

In [ ]:
# now, I just want to see the general picture. I will study it later.
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
from collections import Counter
epochs = 8
#loss_fn = nn.CrossEntropyLoss()
loss_fn = nn.CrossEntropyLoss()

# Ensure model is on the correct device and optimizer is updated
#model.to(device)

In [ ]:
import torch.optim as optim
# now I am going to write the training loop
def train(model, optimizer, loss_fn, train_loader, val_loader, device="cpu"):
    for epoch in range(epochs):
        training_loss = 0.0 # what's the purpose, well those things resets on each epoch.
        valid_loss = 0.0
        correct = 0
        total = 0

        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for batch in train_loader:  # train loader contains batches
            optimizer.zero_grad()   # Clears gradients from the previous batch to prevent accumulation
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            output = model(inputs)
            loss = loss_fn(output, targets)
            loss.backward()
            optimizer.step() #Updates model weights using the computed gradients
            training_loss += loss.item()
            correct += (output.argmax(1) == targets).sum().item()
            total += inputs.size(0)

        training_loss /= len(train_loader)
        train_acc  = 100 * correct / total


        model.eval()
        val_correct, val_total = 0, 0

        with torch.no_grad():
            for batch in val_loader:
                inputs, targets = batch
                inputs = inputs.to(device)
                targets = targets.to(device)
                output = model(inputs)
                loss = loss_fn(output, targets)
                valid_loss += loss.item()

                val_correct += (output.argmax(1) == targets).sum().item()
                val_total   += inputs.size(0)

        valid_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total
        scheduler.step()

        print(f"Epoch {epoch:2d}/{epochs-1} | "
          f"Loss: {training_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Val Acc: {val_acc:.2f}% ")

In [ ]:
# Ensure model is on the correct device
model.to(device)

# Re-initialize the optimizer to ensure it points to the model parameters on the current device
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Run training
# Note: Make sure to execute the cell containing the 'train' function definition first!
train(model, optimizer, torch.nn.CrossEntropyLoss(), train_loader, val_loader, device)

Epoch  0/7 | Loss: 1.0408 | Train Acc: 60.20% | Val Acc: 72.83% 


/tmp/ipykernel_1953/546196590.py:47: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch  1/7 | Loss: 0.7718 | Train Acc: 70.76% | Val Acc: 78.73% 
Epoch  2/7 | Loss: 0.6752 | Train Acc: 75.20% | Val Acc: 77.60% 
Epoch  3/7 | Loss: 0.6115 | Train Acc: 77.63% | Val Acc: 81.93% 
Epoch  4/7 | Loss: 0.5663 | Train Acc: 79.39% | Val Acc: 83.37% 
Epoch  5/7 | Loss: 0.5469 | Train Acc: 80.44% | Val Acc: 83.30% 
Epoch  6/7 | Loss: 0.5096 | Train Acc: 81.62% | Val Acc: 83.17% 
Epoch  7/7 | Loss: 0.4847 | Train Acc: 82.83% | Val Acc: 84.57% 
